# Continuous first-test replay: one-hour pilot

The default now replays **one continuous source hour starting at the first prepared bin with recorded requests** in the first test range. This avoids the empty leading portion. Every request in that selected hour is retained; queues and learning state do not reset between reporting bins. This is a **temporal subset**, not the complete multi-day test range and not random request sampling.

The pilot is for measuring runtime and checking continuity—not evidence that the policy generalizes across the full range. Its selection uses traffic only, not policy outcomes.

FCFS, length-aware, static SLO-aware and frozen RL run at 8×, using your existing checkpoint with no training. Optional adaptation remains separate. The simulator drains only once after the **selected hour's final arrival**. Progress messages show source timestamp, arrived/resolved requests, queue size and elapsed wall time, roughly every 30 seconds between simulator steps, including drain.

Five source minutes are 37.5 simulated seconds at 8×. SLO deadlines stay five simulated seconds. Reporting uses eventual arrival-cohort outcomes; empty-bin rates are undefined. Set `PILOT_SOURCE_MINUTES=None` only if you intentionally want to retry the entire range; the million-request run proved impractical on this laptop. No completion-time guarantee is made even for pilots.


In [ ]:
from pathlib import Path
import sys, subprocess, json, os
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
BRANCH = "burstgpt_3"
REPO_URL = "https://github.com/xfgao210/llm-request-triage.git"
if IS_COLAB:
    REPO = Path("/content/llm-request-triage")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    REPO = Path.cwd().resolve()
    if REPO.name == "notebooks":
        REPO = REPO.parent
assert (REPO / "experiments/continuous_evaluation.py").is_file(), "Use a checkout containing the continuous-evaluation changes."
os.chdir(REPO)
for entry in (str(REPO), str(REPO / "src")):
    if entry not in sys.path:
        sys.path.insert(0, entry)
print("Repository:", REPO)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


In [ ]:
# Optional on an already configured local environment.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[rl,analysis]", "pandas"], check=True)
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Image, FileLink
from experiments import continuous_evaluation as continuous
from experiments.train_dqn import configured_candidate_order
from llm_triage.workload.burstgpt import sha256


## 1. Reuse prepared data and a checkpoint

Point `SOURCE_RUN` to the previous completed **full run folder**, not its ZIP. It must retain `data/processed` and `training_run/training`. No raw CSV upload or retraining is needed. In Colab, point to the equivalent folder on Drive.

Use a new `RUN_NAME`, separate from the interrupted full-range run. `PILOT_SOURCE_MINUTES=60` selects the first hour with recorded traffic; increase duration only after benchmarking. The interrupted policy cannot resume midway. Completed policy cases can be reused only within the same checked configuration. Keep adaptation disabled for the primary frozen-policy comparison. If enabled, it uses fresh Adam/replay state once at range start, epsilon=0, LR=1e-5, warmup=2048, batch=64, one eligible update per 32 decisions, at least 60 simulated seconds and 32 new outcomes since the previous update. The default global cap is **None**, unlike the earlier 100-update short-window budget. This is an explicitly experimental long-run schedule, not a tuned guarantee. Weights/replay/optimizer are not reset at reporting boundaries.


In [ ]:
SOURCE_RUN = str(
    Path("/content/drive/MyDrive/burstgpt3_adaptation_stress/length_aware_drop02_full") if IS_COLAB
    else REPO / "results/runs/colab_burstgpt3_stress/length_aware_drop02_full"
)
RUN_NAME = "continuous_mixed_8x_pilot_1h_v1"
ARRIVAL_MULTIPLIER = 8
PILOT_SOURCE_MINUTES = 60  # None opts into the entire test range (very slow).
PILOT_START_TIMESTAMP_S = None  # Optional prepared-bin-aligned timestamp inside the first test range.
PROGRESS_EVERY_SECONDS = 30.0
SEED = 7
DEVICE = "cpu"
INCLUDE_ADAPTING = False
ONLINE_MIN_SIMULATED_SECONDS = 60.0
ONLINE_MAX_UPDATES = None
REPORT_CADENCES_MINUTES = [5, 10, 15]
DISPLAY_CADENCE_MINUTES = 15
STABILITY_MIN_ARRIVALS = 30
STABILITY_ATTAINMENT_TARGET = 0.8  # Analysis target, NOT a change to request deadlines or reward.
CONFIRM_REPLAY = False  # Review the selected interval and request count, then set True.
EXPORT_REQUEST_DIAGNOSTICS = False  # True can create a very large ZIP.

source = Path(SOURCE_RUN).expanduser().resolve()
TRAINING_DIR = source / "training_run/training"
DATA_DIR = source / "data"
RESULTS_ROOT = Path("/content/drive/MyDrive/burstgpt3_continuous") if IS_COLAB else REPO / "results/runs/burstgpt3_continuous"
assert Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {".", ".."}, "Use a simple, new run name."
RUN_ROOT = RESULTS_ROOT / RUN_NAME
assert DISPLAY_CADENCE_MINUTES in REPORT_CADENCES_MINUTES
print("Results:", RUN_ROOT)


In [ ]:
manifest_path = DATA_DIR / "processed/manifest.json"
manifest = json.loads(manifest_path.read_text())
training_record = json.loads((TRAINING_DIR / "run.json").read_text())
window = continuous.first_test_window(manifest, source_minutes=PILOT_SOURCE_MINUTES, source_start_s=PILOT_START_TIMESTAMP_S)
selected = next(c for c in training_record["checkpoints"] if c["seed"] == SEED)
assert sha256(TRAINING_DIR / f"dqn_seed{SEED}.npz") == selected["sha256"]
assert training_record["manifest_sha256"] == sha256(manifest_path)
assert training_record["selection_split"] == "validation" and not training_record["test_evaluated"]
assert training_record["config"].get("observation", {}).get("include_episode_progress") is False
display(pd.DataFrame([window]))
display(pd.DataFrame([dict(
    requests=window["requests"],
    source_days=(window["end_timestamp_s"]-window["start_timestamp_s"])/86400,
    simulated_hours=(window["end_timestamp_s"]-window["start_timestamp_s"])/ARRIVAL_MULTIPLIER/3600,
    multiplier=ARRIVAL_MULTIPLIER, seed=SEED, checkpoint_episode=selected["episode"],
    candidate_order=configured_candidate_order(training_record["config"]),
    drop_penalty=training_record["reward_config"]["drop"],
    policies=5 if INCLUDE_ADAPTING else 4,
    queue_reset_between_bins=False, request_sampling=False, full_test_range=PILOT_SOURCE_MINUTES is None,
)]))
print("One continuous environment per policy; drain only after the selected interval ends.")
print("Pilot duration (source minutes):", PILOT_SOURCE_MINUTES, "| Progress interval (wall seconds):", PROGRESS_EVERY_SECONDS)
print("Large event traces remain in memory. Keep an eye on RAM and disk.")
print("Optional adapting settings:", INCLUDE_ADAPTING, ONLINE_MIN_SIMULATED_SECONDS, ONLINE_MAX_UPDATES)


## 2. Run the continuous pilot

Set `CONFIRM_REPLAY=True` after reviewing the preflight. All policies receive every request in the selected hour. Reporting boundaries only record metrics; they do not drain queues or reset state. A completion message distinguishes replay from report building.

Progress appears roughly every `PROGRESS_EVERY_SECONDS` wall seconds when simulator control returns between steps. `arrived` counts observed arrivals; `resolved` counts finished/expired/dropped requests; `queue` includes deferred requests. The source clock can exceed the selected interval during the end-only drain. These are diagnostic counters, never policy inputs.

A rerun checks completed policy caches. Interrupted individual policies start again. Duration/settings/code changes require a new name. Do not rerun the old million-request folder.


In [ ]:
assert CONFIRM_REPLAY, "Review preflight, then set CONFIRM_REPLAY=True and rerun parameters/preflight."
summary_rows = continuous.run(
    TRAINING_DIR, DATA_DIR, RUN_ROOT, multiplier=ARRIVAL_MULTIPLIER, seed=SEED,
    include_adapting=INCLUDE_ADAPTING, online_min_seconds=ONLINE_MIN_SIMULATED_SECONDS,
    online_max_updates=ONLINE_MAX_UPDATES, device=DEVICE,
    pilot_source_minutes=PILOT_SOURCE_MINUTES, progress_interval_s=PROGRESS_EVERY_SECONDS,
    pilot_start_timestamp_s=PILOT_START_TIMESTAMP_S,
)
summary = pd.DataFrame(summary_rows)
display(summary[["policy", "arrivals", "slo_attainment", "drop_rate", "unfinished_rate",
                 "window_goodput_rps", "p95_ttft_s", "ttft_target_attainment", "drain_s"]])


## 3. Derive 5-, 10- and 15-minute reports—without replaying

Re-run the following cells to change reporting cadence. Rates and p95 values are recalculated from request outcomes, never averaged across smaller bins. **No simulation or training happens here.**

SLO/drop rates follow requests arriving in the interval. Goodput counts on-time completions actually occurring in the interval, including earlier arrivals. Tail completions are excluded from interval goodput. Queue statistics are duration-weighted iteration snapshots, not cycle-accurate real-GPU counters.

Stability summaries exclude bins below `STABILITY_MIN_ARRIVALS`, explicitly show eligible time/request coverage, and use a predeclared attainment target. A low standard deviation at poor attainment is not evidence of a good policy. Workload changes can also explain rate variation.


In [ ]:
continuous.build_reports(
    RUN_ROOT, DATA_DIR, cadences=REPORT_CADENCES_MINUTES,
    min_arrivals=STABILITY_MIN_ARRIVALS, attainment_target=STABILITY_ATTAINMENT_TARGET,
)
stability = pd.read_csv(RUN_ROOT / "stability.csv")
timeline = pd.read_csv(RUN_ROOT / f"timeline_{DISPLAY_CADENCE_MINUTES}min.csv")
learning = pd.read_csv(RUN_ROOT / "evaluation/learning_summary.csv")
display(stability[stability.cadence_minutes == DISPLAY_CADENCE_MINUTES])
display(learning)


In [ ]:
FIGURES = RUN_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
origin = window["start_timestamp_s"]
fig, axes = plt.subplots(3, 2, figsize=(15, 12), constrained_layout=True)
metrics = [
    ("slo_attainment", "Arrival-cohort SLO attainment", True),
    ("drop_rate", "Arrival-cohort drop rate", True),
    ("completion_goodput_rps", "On-time completions / simulated second", False),
    ("mean_queue", "Mean queue (no resets)", False),
    ("p95_ttft_s", "Arrival-cohort p95 TTFT, observed requests (s)", False),
    ("ttft_target_attainment", "TTFT <= 1s / all arrivals", True),
]
for policy, frame in timeline.groupby("policy"):
    hours = (frame.source_start_s-origin)/3600
    for ax, (metric, title, percent) in zip(axes.flat, metrics):
        ax.plot(hours, frame[metric], label=policy, linewidth=1, marker=".", markersize=2)
        ax.set(title=title, xlabel="Source hours since first test range start")
        if percent:
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            ax.set_ylim(0, 1.05)
        ax.grid(alpha=.2)
axes[0, 0].axhline(STABILITY_ATTAINMENT_TARGET, linestyle="--", color="gray", label="Analysis target")
axes[0, 0].legend(fontsize=8)
fig.suptitle(f"Continuous {window['group']} / {ARRIVAL_MULTIPLIER}x / {DISPLAY_CADENCE_MINUTES}-minute source cohorts")
chart_path = FIGURES / f"continuous_{DISPLAY_CADENCE_MINUTES}min.png"
fig.savefig(chart_path, dpi=150)
plt.show()

reference = timeline[timeline.policy == "fcfs"]
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
hours = (reference.source_start_s-origin)/3600
for ax, metric, label in zip(axes, ["offered_rps", "mean_output_tokens", "mean_input_tokens"],
                           ["Offered requests / simulated s", "Mean recorded output tokens (offline)", "Mean input tokens"]):
    ax.plot(hours, reference[metric], linewidth=.8, marker=".", markersize=2)
    ax.set_ylabel(label)
    ax.grid(alpha=.2)
axes[-1].set_xlabel("Source hours since first test range start")
workload_path = FIGURES / f"workload_{DISPLAY_CADENCE_MINUTES}min.png"
fig.savefig(workload_path, dpi=150)
plt.show()


In [ ]:
import base64, html, zipfile
plan = json.loads((RUN_ROOT / "continuous_plan.json").read_text())
summary = pd.read_csv(RUN_ROOT / "evaluation/summary.csv")
parts = [
    "<!doctype html><html><head><meta charset='utf-8'><title>Continuous first-test pilot/replay</title>",
    "<style>body{font:15px system-ui;margin:30px}img{max-width:100%}table{border-collapse:collapse}td,th{padding:5px;border:1px solid #ccc}pre{white-space:pre-wrap}</style></head><body>",
    "<h1>Continuous first-test pilot/replay</h1>",
    "<p>Every request in the declared interval retained (pilot or full range as recorded below); one empty-queue start per policy; no resets/drains at reporting boundaries. Full drain only at the range end. Source-time reporting; simulated-time deadlines. Exploratory reused test data, one seed.</p>",
    "<p>Empty-bin rates are undefined. TTFT percentiles exclude no-token requests; TTFT-target attainment includes all arrivals. Low variability alone does not imply good performance. Sparse-bin exclusions and coverage appear in the stability table.</p>",
    "<h2>Overall results</h2>", summary.to_html(index=False),
    "<h2>Stability at all reporting cadences</h2>", stability.to_html(index=False),
    "<h2>Learning</h2>", learning.to_html(index=False),
]
for path in (chart_path, workload_path):
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    parts.append(f"<img alt='{html.escape(path.name)}' src='data:image/png;base64,{encoded}'>")
parts += ["<h2>Replay plan</h2><pre>", html.escape(json.dumps(plan, indent=2)), "</pre></body></html>"]
REPORT = RUN_ROOT / "continuous_report.html"
REPORT.write_text("\n".join(parts), encoding="utf-8")
members = [REPORT, RUN_ROOT / "continuous_plan.json", RUN_ROOT / "report_settings.json",
           RUN_ROOT / "stability.csv", RUN_ROOT / "evaluation/run.json",
           RUN_ROOT / "evaluation/summary.csv", RUN_ROOT / "evaluation/learning_summary.csv",
           chart_path, workload_path]
members += [RUN_ROOT / f"timeline_{c}min.csv" for c in REPORT_CADENCES_MINUTES]
for pattern in ("*/load_*/*/summary.json", "*/load_*/*/learning.json", "*/load_*/*/*_updates.csv",
                "*/load_*/*/*_final.npz"):
    members += sorted((RUN_ROOT / "evaluation").glob(pattern))
if EXPORT_REQUEST_DIAGNOSTICS:
    members += sorted((RUN_ROOT / "evaluation").glob("*/load_*/*/request_diagnostics.csv"))
ARCHIVE = RUN_ROOT / (RUN_NAME + "_report.zip")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in members:
        bundle.write(path, path.relative_to(RUN_ROOT).as_posix())
    bundle.write(TRAINING_DIR / "run.json", "source_training/run.json")
    bundle.write(TRAINING_DIR / f"dqn_seed{SEED}.npz", f"source_training/dqn_seed{SEED}.npz")
display(FileLink(str(REPORT)))
display(FileLink(str(ARCHIVE)))
print("Request-level diagnostics stay in the run folder for rebinning; the default report ZIP omits them.")
print("Keep the original source run and prepared data. This report ZIP is not a complete resumable simulation archive.")
